# Structured Output

Models can be requested to provide their responses in a format that matches a given **schema**. This is useful for ensuring that the output can be easily parsed and used in subsequent processing.

LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

**Pydantic models** provide the richest feature set for defining structured output. They support:

- Field validation
- Field descriptions
- Nested data structures

Using Pydantic helps ensure that model responses conform to the expected schema, making them easier to validate and integrate into applications.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000200C47EBCB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000200C4A14830>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel ,Field

class Movie(BaseModel) :
    title:str=Field(description="This is title of Movie")
    actors:list=Field(description="This is list star cast in the movie")
    duration:int=Field(description="This is the duration of movie in  minutes")
    director:str=Field(description="This is name of director of the movie")
    rating:float=Field(description="This is ImDb rating of the movie")

model_with_str = model.with_structured_output(Movie)

In [3]:
model_with_str

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000200C47EBCB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000200C4A14830>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [4]:
model.invoke("Please provide details of movie 'Inception' ")

AIMessage(content='<think>\nOkay, the user is asking for details about the movie "Inception." Let me start by recalling what I know about it. It\'s a 2010 film directed by Christopher Nolan. The main character is Dom Cobb, played by Leonardo DiCaprio. He\'s a thief who enters people\'s dreams to steal secrets. The plot involves a team trying to implant an idea into someone\'s subconscious, which is called "inception." \n\nI should mention the supporting cast: Joseph Gordon-Levitt, Elliot Page, Tom Hardy, and Ken Watanabe. Each plays a significant role in the team. The story is complex, blending action with philosophical themes about reality and dreams. The user might want to know the main plot points, but maybe also some of the underlying concepts like the dream layers and the spinning top.\n\nI need to explain the concept of shared dreaming and the use of the device called the PASIV. The team has different roles: Arthur handles the logistics, Eames is a forger, Ariadne is the architec

In [5]:
res = model_with_str.invoke("Please provide details of movie 'Inception'")

In [6]:
print(res)

title='Inception' actors=['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Ellen Page', 'Tom Hardy'] duration=148 director='Christopher Nolan' rating=8.8


## Message output alongside parsed structure

In [8]:
from pydantic import BaseModel ,Field

class Movie(BaseModel) :
    """A movie with details."""
    title:str=Field(... , description="This is title of Movie")
    year:int=Field(... , description="This is year on which movie was released")
    actors:list=Field(... , description="This is list star cast in the movie")
    duration:int=Field(... , description="This is the duration of movie in  minutes")
    director:str=Field(... , description="This is name of director of the movie")
    rating:float=Field(... , description="This is ImDb rating of the movie")

model_with_str = model.with_structured_output(Movie , include_raw=True)

In [9]:
res = model_with_str.invoke("Please provide details of movie 'Inception'")
res


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie 'Inception'. I need to use the Movie function to get that information. Let me check the required parameters: title, year, actors, duration, director, and rating. \n\nFirst, I know the title is 'Inception'. The release year was 2010. The director is Christopher Nolan. The main actors include Leonardo DiCaprio, Joseph Gordon-Levitt, and Elliot Page. The duration is around 148 minutes. The IMDb rating is about 8.8.\n\nI should structure the function call with all these details. Let me make sure I include all required fields and correct data types. The actors are an array, so I need to list them properly. The rating is a number, so 8.8 without quotes. Year and duration are integers. Alright, that should cover it.\n", 'tool_calls': [{'id': 'h9pnd8fer', 'function': {'arguments': '{"actors":["Leonardo DiCaprio","Joseph Gordon-Levitt","Elliot Page"],"director":"Christophe

## Nested Structure

In [10]:
from pydantic import BaseModel , Field

class Actor(BaseModel):
    name : str
    role:str

class MovieDetails(BaseModel):
    title:str=Field(description="This is title of Movie")
    actors:list[Actor]=Field(description="This is list star cast in the movie")
    genres: list=Field(description="This is list of genres this movie belong to")
    duration:int=Field(description="This is the duration of movie in  minutes")
    budget:float | None=Field(description="This is budget of movie in USD millions")
    director:str=Field(description="This is name of director of the movie")
    rating:float=Field(description="This is ImDb rating of the movie")

model_with_str = model.with_structured_output(MovieDetails)
response = model_with_str.invoke("Provide details about the movie Inception")
response
    

MovieDetails(title='Inception', actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')], genres=['Science Fiction', 'Action', 'Thriller'], duration=148, budget=160.0, director='Christopher Nolan', rating=8.8)

## TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [11]:
from typing_extensions import TypedDict ,Annotated

class MovieDict(TypedDict):
    """"A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withTypedDict_str = model.with_structured_output(MovieDict)
response=model_withTypedDict_str.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [13]:
class Actor(TypedDict):
    name : str
    role:str

class MovieDetails(TypedDict):
    title:str
    actors:list[Actor]
    genres: list
    duration:int
    budget:float | None
    director:str
    rating:float

model_with_str = model.with_structured_output(MovieDetails)
response = model_with_str.invoke("Provide details about the movie Avengers")
response

{'actors': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'budget': 220000000,
 'director': 'Joss Whedon',
 'duration': 143,
 'genres': ['Action', 'Superhero', 'Adventure'],
 'rating': 8,
 'title': 'Avengers'}

In [14]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

## DataClasses


A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [18]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name:str
    address:str
    phone:str

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

res = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
res

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='e92717b8-b009-48d3-94b4-a368f15e3b0d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given string: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function with parameters name, address, and phone. Wait, but the input has an email and a phone number. The function requires name, address, and phone. Hmm, the email isn\'t part of the parameters. Did I miss something?\n\nFirst, I need to parse the input. The name is John Doe. The email is john@example.com, but the function doesn\'t have an email field. The phone number is (555) 123-4567. The function requires address, but there\'s no address provided here. Oh, maybe the user made a mistake? Or perhaps the email was meant to be part of the address? That d